In [ ]:
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap

# paths to the JSON files produced by the final POLAR runs
RESULTS_PATHS = {
    "Iran": "./outputs/iran/polar_youtube_run/per_user_scores.json",
    "Afghanistan": "./outputs/afghanistan/polar_youtube_run/per_user_scores.json",
}

frames = []
for conflict_name, path in RESULTS_PATHS.items():
    with open(path, "r", encoding="utf-8") as f:
        rows = json.load(f)
    tmp = pd.DataFrame(rows)
    tmp["conflict"] = conflict_name
    frames.append(tmp)

df = pd.concat(frames, ignore_index=True)
df = df[df["s"].notna()].copy()
df["s"] = pd.to_numeric(df["s"], errors="coerce")
df = df[df["s"].notna()].copy()
df.head()


In [ ]:
def infer_channel_type(row):
    candidates = [
        row.get("source_kind"),
        row.get("group_label"),
        row.get("corpus"),
        row.get("channel_title"),
        row.get("user_id"),
    ]
    text = " ".join([str(x).lower() for x in candidates if pd.notna(x) and x is not None])

    if "news" in text:
        return "News"
    if "influencer" in text:
        return "Influencer"
    return "Other"


def infer_conflict(row):
    explicit = row.get("conflict")
    if pd.notna(explicit) and explicit is not None:
        return str(explicit)

    candidates = [
        row.get("corpus"),
        row.get("group_label"),
        row.get("channel_title"),
        row.get("user_id"),
    ]
    text = " ".join([str(x).lower() for x in candidates if pd.notna(x) and x is not None])

    if "afghanistan" in text or "afgh" in text:
        return "Afghanistan"
    if "iran" in text:
        return "Iran"
    return "Other"


def pretty_pair_name(x):
    x = str(x).replace("_", " ").strip()
    x = re.sub(r"\s+", " ", x)
    return x.title()


df["conflict"] = df.apply(infer_conflict, axis=1)
df["channel_type"] = df.apply(infer_channel_type, axis=1)
df = df[df["conflict"].isin(["Iran", "Afghanistan"])].copy()
df = df[df["channel_type"].isin(["News", "Influencer"])].copy()
df["pair_pretty"] = df["pair"].map(pretty_pair_name)

# optional: set your preferred order explicitly here
PAIR_ORDER = []

if not PAIR_ORDER:
    conflict_rank = (
        df.groupby(["pair_pretty", "conflict"])["s"]
        .median()
        .unstack("conflict")
        .assign(conflict_gap=lambda x: (x["Iran"] - x["Afghanistan"]).abs())
        .sort_values(["conflict_gap", "Iran"], ascending=[False, True])
    )
    PAIR_ORDER = conflict_rank.index.tolist()

df = df[df["pair_pretty"].isin(PAIR_ORDER)].copy()
pair_to_y = {pair: i for i, pair in enumerate(PAIR_ORDER)}
df["y0"] = df["pair_pretty"].map(pair_to_y)

# conflict is the main split; channel type is secondary within conflict
conflict_offset_map = {"Iran": -0.18, "Afghanistan": 0.18}
channel_offset_map = {"News": -0.06, "Influencer": 0.06}
df["y"] = df["y0"] + df["conflict"].map(conflict_offset_map) + df["channel_type"].map(channel_offset_map)

# deterministic jitter
rng = np.random.default_rng(42)
df["y"] = df["y"] + rng.normal(0, 0.02, size=len(df))

# conflict medians are the main guide
med = (
    df.groupby(["pair_pretty", "conflict"], as_index=False)["s"]
    .median()
    .rename(columns={"s": "median_s"})
)
med["y0"] = med["pair_pretty"].map(pair_to_y)
med["y_mid"] = med["y0"] + med["conflict"].map(conflict_offset_map)

summary = (
    df.groupby(["pair_pretty", "conflict", "channel_type"], as_index=False)
    .agg(
        median_s=("s", "median"),
        mean_s=("s", "mean"),
        n=("s", "size"),
    )
)
summary = summary.sort_values(["pair_pretty", "conflict", "channel_type"]).reset_index(drop=True)
summary


In [ ]:
# style
conflict_colors = {"Iran": "#c44e52", "Afghanistan": "#4c72b0"}
marker_map = {"News": "o", "Influencer": "^"}
label_map = {
    ("Iran", "News"): "Iran news",
    ("Iran", "Influencer"): "Iran influencers",
    ("Afghanistan", "News"): "Afghanistan news",
    ("Afghanistan", "Influencer"): "Afghanistan influencers",
}
band_cmap = LinearSegmentedColormap.from_list(
    "polar_bands",
    ["#b8c0ff", "#c9b5d8", "#d9b7c8", "#e8c0be", "#f1b28a", "#ee9c6b"]
)

n_pairs = len(PAIR_ORDER)
fig_h = max(5.5, 0.95 * n_pairs + 1.4)
fig, ax = plt.subplots(figsize=(9.4, fig_h), dpi=160)

# background category bands
band_colors = [band_cmap(i / max(n_pairs - 1, 1)) for i in range(n_pairs)]
for i, pair in enumerate(PAIR_ORDER):
    ax.axhspan(i - 0.5, i + 0.5, color=band_colors[i], alpha=0.72, zorder=0)
    ax.hlines(i + 0.5, xmin=-10, xmax=10, color="black", lw=0.8, alpha=0.5, zorder=1)

# zero line
ax.axvline(0, color="black", lw=1.1, ls="--", zorder=2)

# thicker conflict medians highlight the war-to-war comparison
for _, r in med.iterrows():
    color = conflict_colors[r["conflict"]]
    ax.vlines(
        r["median_s"],
        r["y_mid"] - 0.14,
        r["y_mid"] + 0.14,
        color=color,
        lw=2.4,
        linestyles=(0, (4, 3)),
        alpha=0.95,
        zorder=3,
    )

# scatter points: color = conflict, marker = channel type
for conflict in ["Iran", "Afghanistan"]:
    for channel_type in ["News", "Influencer"]:
        sub = df[(df["conflict"] == conflict) & (df["channel_type"] == channel_type)]
        ax.scatter(
            sub["s"],
            sub["y"],
            s=42 if channel_type == "Influencer" else 34,
            marker=marker_map[channel_type],
            facecolor=conflict_colors[conflict],
            edgecolor="black",
            linewidth=0.55,
            alpha=0.92,
            zorder=4,
            label=label_map[(conflict, channel_type)],
        )

# axes, ticks, grid
xpad = max(0.15, 0.08 * (df["s"].max() - df["s"].min()))
xmin = np.floor((df["s"].min() - xpad) * 2) / 2
xmax = np.ceil((df["s"].max() + xpad) * 2) / 2

ax.set_xlim(xmin, xmax)
ax.set_ylim(-0.5, n_pairs - 0.5)
ax.set_yticks(range(n_pairs))
ax.set_yticklabels(PAIR_ORDER, fontsize=11, fontweight="bold")
ax.invert_yaxis()

ax.set_xlabel("Association Score", fontsize=15, fontweight="bold")
ax.set_ylabel("Category", fontsize=15, fontweight="bold")
ax.set_title("POLAR Scores by Conflict and Channel Type", fontsize=16, fontweight="bold", pad=14)

ax.grid(axis="x", which="major", linestyle="--", linewidth=0.8, alpha=0.55, color="gray")
ax.set_axisbelow(True)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

# legend
handles = [
    Line2D([0], [0], marker="o", color="none", markerfacecolor="white", markeredgecolor="black", markeredgewidth=0.8, markersize=7, label="News channels"),
    Line2D([0], [0], marker="^", color="none", markerfacecolor="white", markeredgecolor="black", markeredgewidth=0.8, markersize=7.5, label="Influencer channels"),
    Line2D([0], [0], color="black", lw=1.1, ls="--", label="Zero"),
    Line2D([0], [0], color=conflict_colors["Iran"], lw=2.4, ls=(0, (4, 3)), label="Iran median"),
    Line2D([0], [0], color=conflict_colors["Afghanistan"], lw=2.4, ls=(0, (4, 3)), label="Afghanistan median"),
]
ax.legend(handles=handles, loc="lower left", frameon=True, fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
# quick conflict-first summary table
conflict_summary = (
    df.groupby(["pair_pretty", "conflict"])["s"]
    .agg(["median", "mean", "count"])
    .reset_index()
)
conflict_summary = conflict_summary.sort_values(["pair_pretty", "conflict"]).reset_index(drop=True)
conflict_summary


In [ ]:
# print every channel name with its embedding coordinates
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

RUN_DIRS = {
    "Iran": Path("./outputs/iran/polar_youtube_run"),
    "Afghanistan": Path("./outputs/afghanistan/polar_youtube_run"),
}
def _l2_normalize_rows(x):
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1.0, norms)
    return x / norms

def load_channel_embeddings(conflict_name, run_dir):
    users = json.loads((run_dir / "users.json").read_text(encoding="utf-8"))
    tok = AutoTokenizer.from_pretrained(run_dir / "model")
    model = AutoModelForMaskedLM.from_pretrained(run_dir / "model")
    model.eval()

    with torch.no_grad():
        weights = model.get_input_embeddings().weight.detach().cpu().numpy().astype("float32")
    weights = _l2_normalize_rows(weights)
    vocab = tok.get_vocab()

    rows = []
    vecs = []
    for row in users:
        token = row.get("token")
        tok_id = vocab.get(token)
        if tok_id is None:
            continue

        rows.append(
            {
                "conflict": conflict_name,
                "channel_id": row.get("channel_id"),
                "channel_title": row.get("channel_title") or row.get("user_id") or token,
                "user_id": row.get("user_id"),
                "token": token,
            }
        )
        vecs.append(weights[tok_id])

    return pd.DataFrame(rows), np.vstack(vecs)

channel_blocks = []
all_vecs = []
for conflict_name, run_dir in RUN_DIRS.items():
    meta_df, vecs = load_channel_embeddings(conflict_name, run_dir)
    channel_blocks.append((meta_df, vecs))
    all_vecs.append(vecs)

coord_cols = [f"dim_{i}" for i in range(all_vecs[0].shape[1])]
global_vecs = np.vstack(all_vecs)
dim_variances = global_vecs.var(axis=0)
top_dim_idx = np.argsort(dim_variances)[::-1][:2]
selected_coord_cols = [coord_cols[i] for i in top_dim_idx]

channel_frames = []
for meta_df, vecs in channel_blocks:
    coords_df = pd.DataFrame(vecs[:, top_dim_idx], columns=selected_coord_cols)
    channel_frames.append(pd.concat([meta_df.reset_index(drop=True), coords_df], axis=1))

channel_coordinates = pd.concat(channel_frames, ignore_index=True)
channel_coordinates

for conflict_name, sub in channel_coordinates.groupby("conflict", sort=False):
    print(f"\n{conflict_name} channel coordinates:")
    print(sub[["channel_title", *selected_coord_cols]].to_string(index=False))


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D
from matplotlib.patches import Ellipse
from matplotlib.ticker import AutoMinorLocator
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

try:
    import umap.umap_ as umap
    HAVE_UMAP = True
except Exception:
    umap = None
    HAVE_UMAP = False

RUN_DIRS = {
    "Iran": Path("./outputs/iran/polar_youtube_run"),
    "Afghanistan": Path("./outputs/afghanistan/polar_youtube_run"),
}

def _l2_normalize_rows(x):
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1.0, norms)
    return x / norms

def _confidence_ellipse(ax, x, y, color, n_std=1.55, alpha=0.10, lw=2.0):
    if len(x) < 3:
        return
    cov = np.cov(x, y)
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(np.maximum(vals, 1e-9))
    ell = Ellipse(
        xy=(np.mean(x), np.mean(y)),
        width=width,
        height=height,
        angle=angle,
        facecolor=color,
        edgecolor=color,
        linewidth=lw,
        alpha=alpha,
        zorder=1,
    )
    ax.add_patch(ell)

def load_channel_embeddings(conflict_name, run_dir):
    users = json.loads((run_dir / "users.json").read_text(encoding="utf-8"))
    tok = AutoTokenizer.from_pretrained(run_dir / "model")
    model = AutoModelForMaskedLM.from_pretrained(run_dir / "model")
    model.eval()

    with torch.no_grad():
        weights = model.get_input_embeddings().weight.detach().cpu().numpy().astype("float32")
    weights = _l2_normalize_rows(weights)
    vocab = tok.get_vocab()

    rows = []
    vecs = []
    for row in users:
        token = row.get("token")
        tok_id = vocab.get(token)
        if tok_id is None:
            continue

        source_text = " ".join(
            str(row.get(k, "")).lower()
            for k in ["source_kind", "label_majority", "group_label", "channel_title", "user_id"]
        )
        channel_type = "News" if "news" in source_text else "Influencer"

        rows.append(
            {
                "conflict": conflict_name,
                "channel_type": channel_type,
                "channel_id": row.get("channel_id"),
                "channel_title": row.get("channel_title"),
                "user_id": row.get("user_id"),
                "token": token,
                "n_posts": int(row.get("n_posts", 0) or 0),
                "n_videos": int(row.get("n_videos", 0) or 0),
            }
        )
        vecs.append(weights[tok_id])

    return pd.DataFrame(rows), np.vstack(vecs)

# ---------------------------------------------------------
# build embedding_df
# ---------------------------------------------------------
frames = []
vec_blocks = []

for conflict_name, run_dir in RUN_DIRS.items():
    meta_block, vec_block = load_channel_embeddings(conflict_name, run_dir)
    frames.append(meta_block)
    vec_blocks.append(vec_block)

embedding_df = pd.concat(frames, ignore_index=True)
X = _l2_normalize_rows(np.vstack(vec_blocks))

# ---------------------------------------------------------
# projection
# ---------------------------------------------------------
pca_dims = max(2, min(25, X.shape[0] - 1, X.shape[1]))
X_pca = PCA(n_components=pca_dims, random_state=42).fit_transform(X)

if HAVE_UMAP:
    n_neighbors = min(10, X.shape[0] - 1)
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=n_neighbors,
        min_dist=0.45,
        metric="cosine",
        init=X_pca[:, :2],
        random_state=42,
    )
    coords = reducer.fit_transform(X)
    reducer_label = f"UMAP (cosine metric, PCA init, n_neighbors={n_neighbors})"
else:
    perplexity = max(5, min(12, (X.shape[0] - 1) // 3))
    coords = TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        metric="cosine",
        random_state=42,
    ).fit_transform(X_pca)
    reducer_label = f"t-SNE fallback (cosine metric, perplexity={perplexity})"

embedding_df["x"] = coords[:, 0]
embedding_df["y"] = coords[:, 1]
embedding_df["size"] = 110 + 24 * np.log1p(embedding_df["n_posts"].clip(lower=1))

# ---------------------------------------------------------
# build shift_df
# ---------------------------------------------------------
shared_rows = []
for channel_id, g in embedding_df.groupby("channel_id"):
    if g["conflict"].nunique() != 2:
        continue
    g = g.sort_values("conflict").reset_index(drop=True)
    shared_rows.append(
        {
            "channel_id": channel_id,
            "channel_title": g.loc[0, "channel_title"],
            "proj_shift": float(np.hypot(g.loc[0, "x"] - g.loc[1, "x"], g.loc[0, "y"] - g.loc[1, "y"])),
        }
    )

shift_df = pd.DataFrame(shared_rows).sort_values("proj_shift", ascending=False).reset_index(drop=True)

# ---------------------------------------------------------
# plot
# ---------------------------------------------------------
FIG_W = 20
FIG_H = 8
FIG_DPI = 220

FS_LABEL = 22
FS_TICK = 18
FS_LEGEND = 18
FS_REDUCER = 16

POINT_EDGE_LW = 0.9
LINE_LW = 1.2

conflict_colors = {"Iran": "#c44e52", "Afghanistan": "#4c72b0"}
marker_map = {"News": "o", "Influencer": "^"}

fig, ax = plt.subplots(
    figsize=(FIG_W, FIG_H),
    dpi=FIG_DPI,
    constrained_layout=True
)
fig.patch.set_facecolor("white")
ax.set_facecolor("#fcfcfe")

x_span = embedding_df["x"].max() - embedding_df["x"].min()
y_span = embedding_df["y"].max() - embedding_df["y"].min()
x_pad = 0.10 * x_span if x_span > 0 else 1.0
y_pad = 0.10 * y_span if y_span > 0 else 1.0

ax.set_xlim(embedding_df["x"].min() - x_pad, embedding_df["x"].max() + x_pad)
ax.set_ylim(embedding_df["y"].min() - y_pad, embedding_df["y"].max() + y_pad)

for _, row in shift_df.iterrows():
    pair = embedding_df[embedding_df["channel_id"].eq(row["channel_id"])].sort_values("conflict")
    ax.plot(
        pair["x"], pair["y"],
        color="#7f7f7f",
        lw=LINE_LW,
        alpha=0.58,
        zorder=1,
        solid_capstyle="round",
    )

for conflict, color in conflict_colors.items():
    sub = embedding_df[embedding_df["conflict"].eq(conflict)]
    _confidence_ellipse(ax, sub["x"], sub["y"], color=color, n_std=1.55, alpha=0.10, lw=2.0)
    ax.scatter(
        sub["x"].mean(),
        sub["y"].mean(),
        s=340,
        marker="X",
        c=color,
        edgecolors="white",
        linewidths=1.9,
        zorder=4,
    )

for conflict in ["Iran", "Afghanistan"]:
    for channel_type in ["News", "Influencer"]:
        sub = embedding_df[
            embedding_df["conflict"].eq(conflict) &
            embedding_df["channel_type"].eq(channel_type)
        ]

        ax.scatter(
            sub["x"],
            sub["y"],
            s=sub["size"],
            c=conflict_colors[conflict],
            marker=marker_map[channel_type],
            edgecolors="black",
            linewidths=POINT_EDGE_LW,
            alpha=1.0,
            zorder=3,
        )

ax.text(
    0.015,
    0.02,
    reducer_label,
    transform=ax.transAxes,
    fontsize=FS_REDUCER,
    color="#444444",
    bbox=dict(
        boxstyle="round,pad=0.30",
        facecolor="white",
        edgecolor="#d0d7de",
        linewidth=0.9,
        alpha=0.97,
    ),
)

ax.set_xlabel("Projection axis 1", fontsize=FS_LABEL, fontweight="bold")
ax.set_ylabel("Projection axis 2", fontsize=FS_LABEL, fontweight="bold")

ax.xaxis.set_minor_locator(AutoMinorLocator(4))
ax.yaxis.set_minor_locator(AutoMinorLocator(4))

ax.grid(which="major", color="#b8c0cc", linewidth=0.85, alpha=0.48)
ax.grid(which="minor", color="#d7dde6", linewidth=0.60, alpha=0.60)

ax.set_axisbelow(True)
ax.tick_params(axis="both", which="major", labelsize=FS_TICK, width=1.2, length=7.5)
ax.tick_params(axis="both", which="minor", width=0.8, length=4.5)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

for spine in ["left", "bottom"]:
    ax.spines[spine].set_color("#b8bec9")
    ax.spines[spine].set_linewidth(1.0)

legend_handles = [
    Line2D([0], [0], marker="o", linestyle="", markerfacecolor=conflict_colors["Iran"],
           markeredgecolor="black", markeredgewidth=0.8, markersize=11, label="Iran"),
    Line2D([0], [0], marker="o", linestyle="", markerfacecolor=conflict_colors["Afghanistan"],
           markeredgecolor="black", markeredgewidth=0.8, markersize=11, label="Afghanistan"),
    Line2D([0], [0], marker="o", linestyle="", markerfacecolor="white",
           markeredgecolor="black", markeredgewidth=1.0, markersize=11, label="News"),
    Line2D([0], [0], marker="^", linestyle="", markerfacecolor="white",
           markeredgecolor="black", markeredgewidth=1.0, markersize=11.5, label="Influencer"),
    Line2D([0], [0], color="#7f7f7f", lw=1.3, alpha=0.75, label="Same channel across wars"),
    Line2D([0], [0], marker="X", linestyle="", markerfacecolor="#666666",
           markeredgecolor="white", markeredgewidth=1.0, markersize=12, label="Conflict centroid"),
]

ax.legend(
    handles=legend_handles,
    loc="upper right",
    bbox_to_anchor=(0.985, 0.985),
    borderaxespad=0.0,
    frameon=True,
    fancybox=True,
    framealpha=0.97,
    facecolor="white",
    edgecolor="#d0d7de",
    fontsize=FS_LEGEND,
)

# ---------------------------------------------------------
# save figure as PDF
# ---------------------------------------------------------
output_dir = Path("images")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "embedding_map.pdf"

plt.savefig(
    output_path,
    format="pdf",
    bbox_inches="tight",
    facecolor=fig.get_facecolor(),
)

print(f"Saved figure to: {output_path.resolve()}")

plt.show()

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D
from matplotlib.patches import Ellipse
from matplotlib.ticker import AutoMinorLocator
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

try:
    import umap.umap_ as umap
    HAVE_UMAP = True
except Exception:
    umap = None
    HAVE_UMAP = False

RUN_DIRS = {
    "Iran": Path("./outputs/iran/polar_youtube_run"),
    "Afghanistan": Path("./outputs/afghanistan/polar_youtube_run"),
}

def _l2_normalize_rows(x):
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1.0, norms)
    return x / norms

def _confidence_ellipse(ax, x, y, color, n_std=1.55, alpha=0.10, lw=2.0):
    if len(x) < 3:
        return
    cov = np.cov(x, y)
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[order]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(np.maximum(vals, 1e-9))
    ell = Ellipse(
        xy=(np.mean(x), np.mean(y)),
        width=width,
        height=height,
        angle=angle,
        facecolor=color,
        edgecolor=color,
        linewidth=lw,
        alpha=alpha,
        zorder=1,
    )
    ax.add_patch(ell)

def load_channel_embeddings(conflict_name, run_dir):
    users = json.loads((run_dir / "users.json").read_text(encoding="utf-8"))
    tok = AutoTokenizer.from_pretrained(run_dir / "model")
    model = AutoModelForMaskedLM.from_pretrained(run_dir / "model")
    model.eval()

    with torch.no_grad():
        weights = model.get_input_embeddings().weight.detach().cpu().numpy().astype("float32")
    weights = _l2_normalize_rows(weights)
    vocab = tok.get_vocab()

    rows = []
    vecs = []
    for row in users:
        token = row.get("token")
        tok_id = vocab.get(token)
        if tok_id is None:
            continue

        source_text = " ".join(
            str(row.get(k, "")).lower()
            for k in ["source_kind", "label_majority", "group_label", "channel_title", "user_id"]
        )
        channel_type = "News" if "news" in source_text else "Influencer"

        rows.append(
            {
                "conflict": conflict_name,
                "channel_type": channel_type,
                "channel_id": row.get("channel_id"),
                "channel_title": row.get("channel_title"),
                "user_id": row.get("user_id"),
                "token": token,
                "n_posts": int(row.get("n_posts", 0) or 0),
                "n_videos": int(row.get("n_videos", 0) or 0),
            }
        )
        vecs.append(weights[tok_id])

    return pd.DataFrame(rows), np.vstack(vecs)

# ---------------------------------------------------------
# build embedding_df
# ---------------------------------------------------------
frames = []
vec_blocks = []

for conflict_name, run_dir in RUN_DIRS.items():
    meta_block, vec_block = load_channel_embeddings(conflict_name, run_dir)
    frames.append(meta_block)
    vec_blocks.append(vec_block)

embedding_df = pd.concat(frames, ignore_index=True)
X = _l2_normalize_rows(np.vstack(vec_blocks))

# ---------------------------------------------------------
# projection
# ---------------------------------------------------------
pca_dims = max(2, min(25, X.shape[0] - 1, X.shape[1]))
X_pca = PCA(n_components=pca_dims, random_state=42).fit_transform(X)

if HAVE_UMAP:
    n_neighbors = min(10, X.shape[0] - 1)
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=n_neighbors,
        min_dist=0.45,
        metric="cosine",
        init=X_pca[:, :2],
        random_state=42,
    )
    coords = reducer.fit_transform(X)
    reducer_label = f"UMAP (cosine metric, PCA init, n_neighbors={n_neighbors})"
else:
    perplexity = max(5, min(12, (X.shape[0] - 1) // 3))
    coords = TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        metric="cosine",
        random_state=42,
    ).fit_transform(X_pca)
    reducer_label = f"t-SNE fallback (cosine metric, perplexity={perplexity})"

embedding_df["x"] = coords[:, 0]
embedding_df["y"] = coords[:, 1]
embedding_df["size"] = 110 + 24 * np.log1p(embedding_df["n_posts"].clip(lower=1))

# ---------------------------------------------------------
# build shift_df
# ---------------------------------------------------------
shared_rows = []
for channel_id, g in embedding_df.groupby("channel_id"):
    if g["conflict"].nunique() != 2:
        continue
    g = g.sort_values("conflict").reset_index(drop=True)
    shared_rows.append(
        {
            "channel_id": channel_id,
            "channel_title": g.loc[0, "channel_title"],
            "proj_shift": float(np.hypot(g.loc[0, "x"] - g.loc[1, "x"], g.loc[0, "y"] - g.loc[1, "y"])),
        }
    )

shift_df = pd.DataFrame(shared_rows).sort_values("proj_shift", ascending=False).reset_index(drop=True)

# ---------------------------------------------------------
# plot
# ---------------------------------------------------------
FIG_W = 22
FIG_H = 10
FIG_DPI = 220

FS_LABEL = 22
FS_TICK = 18
FS_LEGEND = 18
FS_REDUCER = 16
FS_TEXT = 9

POINT_EDGE_LW = 0.9
LINE_LW = 1.2

conflict_colors = {"Iran": "#c44e52", "Afghanistan": "#4c72b0"}
marker_map = {"News": "o", "Influencer": "^"}

fig, ax = plt.subplots(
    figsize=(FIG_W, FIG_H),
    dpi=FIG_DPI,
    constrained_layout=True
)
fig.patch.set_facecolor("white")
ax.set_facecolor("#fcfcfe")

x_span = embedding_df["x"].max() - embedding_df["x"].min()
y_span = embedding_df["y"].max() - embedding_df["y"].min()
x_pad = 0.10 * x_span if x_span > 0 else 1.0
y_pad = 0.10 * y_span if y_span > 0 else 1.0

ax.set_xlim(embedding_df["x"].min() - x_pad, embedding_df["x"].max() + x_pad)
ax.set_ylim(embedding_df["y"].min() - y_pad, embedding_df["y"].max() + y_pad)

for _, row in shift_df.iterrows():
    pair = embedding_df[embedding_df["channel_id"].eq(row["channel_id"])].sort_values("conflict")
    ax.plot(
        pair["x"], pair["y"],
        color="#7f7f7f",
        lw=LINE_LW,
        alpha=0.58,
        zorder=1,
        solid_capstyle="round",
    )

for conflict, color in conflict_colors.items():
    sub = embedding_df[embedding_df["conflict"].eq(conflict)]
    _confidence_ellipse(ax, sub["x"], sub["y"], color=color, n_std=1.55, alpha=0.10, lw=2.0)
    ax.scatter(
        sub["x"].mean(),
        sub["y"].mean(),
        s=340,
        marker="X",
        c=color,
        edgecolors="white",
        linewidths=1.9,
        zorder=4,
    )

for conflict in ["Iran", "Afghanistan"]:
    for channel_type in ["News", "Influencer"]:
        sub = embedding_df[
            embedding_df["conflict"].eq(conflict) &
            embedding_df["channel_type"].eq(channel_type)
        ]

        ax.scatter(
            sub["x"],
            sub["y"],
            s=sub["size"],
            c=conflict_colors[conflict],
            marker=marker_map[channel_type],
            edgecolors="black",
            linewidths=POINT_EDGE_LW,
            alpha=1.0,
            zorder=3,
        )

# ---------------------------------------------------------
# print all labels
# ---------------------------------------------------------
text_dx = 0.012 * x_span if x_span > 0 else 0.05
text_dy = 0.012 * y_span if y_span > 0 else 0.05

for _, row in embedding_df.iterrows():
    label = str(row["channel_title"]) if pd.notna(row["channel_title"]) else str(row["user_id"])
    ax.text(
        row["x"] + text_dx,
        row["y"] + text_dy,
        label,
        fontsize=FS_TEXT,
        color="black",
        alpha=0.95,
        zorder=5,
        ha="left",
        va="bottom",
        bbox=dict(
            boxstyle="round,pad=0.15",
            facecolor="white",
            edgecolor="none",
            alpha=0.65,
        ),
    )

ax.text(
    0.015,
    0.02,
    reducer_label,
    transform=ax.transAxes,
    fontsize=FS_REDUCER,
    color="#444444",
    bbox=dict(
        boxstyle="round,pad=0.30",
        facecolor="white",
        edgecolor="#d0d7de",
        linewidth=0.9,
        alpha=0.97,
    ),
)

ax.set_xlabel("Projection axis 1", fontsize=FS_LABEL, fontweight="bold")
ax.set_ylabel("Projection axis 2", fontsize=FS_LABEL, fontweight="bold")

ax.xaxis.set_minor_locator(AutoMinorLocator(4))
ax.yaxis.set_minor_locator(AutoMinorLocator(4))

ax.grid(which="major", color="#b8c0cc", linewidth=0.85, alpha=0.48)
ax.grid(which="minor", color="#d7dde6", linewidth=0.60, alpha=0.60)

ax.set_axisbelow(True)
ax.tick_params(axis="both", which="major", labelsize=FS_TICK, width=1.2, length=7.5)
ax.tick_params(axis="both", which="minor", width=0.8, length=4.5)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

for spine in ["left", "bottom"]:
    ax.spines[spine].set_color("#b8bec9")
    ax.spines[spine].set_linewidth(1.0)

legend_handles = [
    Line2D([0], [0], marker="o", linestyle="", markerfacecolor=conflict_colors["Iran"],
           markeredgecolor="black", markeredgewidth=0.8, markersize=11, label="Iran"),
    Line2D([0], [0], marker="o", linestyle="", markerfacecolor=conflict_colors["Afghanistan"],
           markeredgecolor="black", markeredgewidth=0.8, markersize=11, label="Afghanistan"),
    Line2D([0], [0], marker="o", linestyle="", markerfacecolor="white",
           markeredgecolor="black", markeredgewidth=1.0, markersize=11, label="News"),
    Line2D([0], [0], marker="^", linestyle="", markerfacecolor="white",
           markeredgecolor="black", markeredgewidth=1.0, markersize=11.5, label="Influencer"),
    Line2D([0], [0], color="#7f7f7f", lw=1.3, alpha=0.75, label="Same channel across wars"),
    Line2D([0], [0], marker="X", linestyle="", markerfacecolor="#666666",
           markeredgecolor="white", markeredgewidth=1.0, markersize=12, label="Conflict centroid"),
]

ax.legend(
    handles=legend_handles,
    loc="upper right",
    bbox_to_anchor=(0.985, 0.985),
    borderaxespad=0.0,
    frameon=True,
    fancybox=True,
    framealpha=0.97,
    facecolor="white",
    edgecolor="#d0d7de",
    fontsize=FS_LEGEND,
)

# ---------------------------------------------------------
# save figure as PDF
# ---------------------------------------------------------
output_dir = Path("images")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "embedding_map_with_all_labels.pdf"

plt.savefig(
    output_path,
    format="pdf",
    bbox_inches="tight",
    facecolor=fig.get_facecolor(),
)

print(f"Saved figure to: {output_path.resolve()}")

plt.show()

In [ ]:
# =========================================================
# TOPIC SENTIMENT PLOT
# one row per topic, one point per (conflict x channel type)
# x = mean topic sentiment z-score
# color = conflict
# marker = channel type
# rows sorted by overall mean, with strong separation + gradient
# x-axis fixed to [-0.6, 0.6]
# no y-axis title
# no "Zero" in legend
# =========================================================
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import AutoMinorLocator
from matplotlib import cm

TOPIC_RESULTS_PATHS = {
    "Iran": "./outputs/iran/polar_youtube_run/per_user_topic_sentiment_scores.json",
    "Afghanistan": "./outputs/afghanistan/polar_youtube_run/per_user_topic_sentiment_scores.json",
}

TOPIC_ORDER = ["Iran", "Afghanistan", "Trump", "Biden", "War", "Peace", "America", "Terror"]

conflict_colors = {
    "Iran": "#c44e52",
    "Afghanistan": "#4c72b0",
}
marker_map = {"News": "o", "Influencer": "^"}

# ---------------------------------------------------------
# load and clean
# ---------------------------------------------------------
frames = []
for conflict_name, path in TOPIC_RESULTS_PATHS.items():
    with open(path, "r", encoding="utf-8") as f:
        rows = json.load(f)
    tmp = pd.DataFrame(rows)
    tmp["conflict"] = conflict_name
    frames.append(tmp)

topic_df = pd.concat(frames, ignore_index=True)
topic_df["topic_sentiment_score"] = pd.to_numeric(topic_df["topic_sentiment_score"], errors="coerce")
topic_df["topic_sentiment_zscore_within_topic"] = pd.to_numeric(
    topic_df["topic_sentiment_zscore_within_topic"], errors="coerce"
)

topic_df = topic_df[topic_df["topic_sentiment_zscore_within_topic"].notna()].copy()
topic_df["channel_type"] = topic_df.apply(infer_channel_type, axis=1)
topic_df["conflict"] = topic_df.apply(infer_conflict, axis=1)
topic_df["topic_pretty"] = topic_df["topic_label"].fillna(topic_df["topic"]).astype(str).str.title()

topic_df = topic_df[topic_df["topic_pretty"].isin(TOPIC_ORDER)].copy()
topic_df = topic_df[topic_df["channel_type"].isin(["News", "Influencer"])].copy()

# optional: one point per user-topic-conflict
topic_df = (
    topic_df.sort_values(["topic_pretty", "user_id"])
    .drop_duplicates(subset=["topic_pretty", "user_id", "conflict"], keep="first")
    .copy()
)

# ---------------------------------------------------------
# aggregate means
# ---------------------------------------------------------
mean_df = (
    topic_df
    .groupby(["topic_pretty", "conflict", "channel_type"], as_index=False)
    ["topic_sentiment_zscore_within_topic"]
    .mean()
    .rename(columns={"topic_sentiment_zscore_within_topic": "mean_z"})
)

# overall row mean across the 4 displayed group means
row_mean_df = (
    mean_df.groupby("topic_pretty", as_index=False)["mean_z"]
    .mean()
    .rename(columns={"mean_z": "row_mean"})
    .sort_values("row_mean", ascending=False)
    .reset_index(drop=True)
)

sorted_topics = row_mean_df["topic_pretty"].tolist()

# ---------------------------------------------------------
# stronger row spacing
# ---------------------------------------------------------
ROW_STEP = 1.45
topic_to_y = {topic: i * ROW_STEP for i, topic in enumerate(sorted_topics)}
mean_df["y0"] = mean_df["topic_pretty"].map(topic_to_y)

offset_map = {
    ("Iran", "News"): -0.24,
    ("Iran", "Influencer"): -0.08,
    ("Afghanistan", "News"): 0.08,
    ("Afghanistan", "Influencer"): 0.24,
}
mean_df["y"] = mean_df.apply(
    lambda r: r["y0"] + offset_map[(r["conflict"], r["channel_type"])],
    axis=1
)

# ---------------------------------------------------------
# row gradient colors
# ---------------------------------------------------------
cmap = cm.get_cmap("Purples")
row_fill = {}
n_rows = max(1, len(sorted_topics) - 1)

for i, topic in enumerate(sorted_topics):
    t = i / n_rows if n_rows > 0 else 0.0
    row_fill[topic] = cmap(0.12 + 0.22 * t)

# ---------------------------------------------------------
# figure
# ---------------------------------------------------------
FIG_W = 13.5
FIG_H = max(8.5, 1.0 + 0.95 * len(sorted_topics))
FIG_DPI = 220

FS_LABEL = 17
FS_TICK = 13
FS_LEGEND = 12

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H), dpi=FIG_DPI, constrained_layout=True)
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# ---------------------------------------------------------
# strong row bands
# ---------------------------------------------------------
band_halfheight = 0.48
for topic in sorted_topics:
    y0 = topic_to_y[topic]
    ax.axhspan(
        y0 - band_halfheight,
        y0 + band_halfheight,
        color=row_fill[topic],
        alpha=1.0,
        zorder=0,
    )
    ax.hlines(
        [y0 - band_halfheight, y0 + band_halfheight],
        xmin=-100,
        xmax=100,
        color="#9aa3ad",
        lw=1.0,
        alpha=0.9,
        zorder=1,
    )

# zero line
ax.axvline(0, color="black", lw=1.2, ls="--", zorder=2)

# faint center line inside each row
for topic in sorted_topics:
    y0 = topic_to_y[topic]
    ax.hlines(y0, xmin=-100, xmax=100, color="#7e8792", lw=0.8, alpha=0.45, zorder=1)

# ---------------------------------------------------------
# plot means only
# ---------------------------------------------------------
for conflict in ["Iran", "Afghanistan"]:
    for channel_type in ["News", "Influencer"]:
        sub = mean_df[
            (mean_df["conflict"] == conflict) &
            (mean_df["channel_type"] == channel_type)
        ]
        if sub.empty:
            continue

        ax.scatter(
            sub["mean_z"],
            sub["y"],
            s=170 if channel_type == "Influencer" else 150,
            marker=marker_map[channel_type],
            c=conflict_colors[conflict],
            edgecolors="black",
            linewidths=1.0,
            alpha=1.0,
            zorder=4,
        )

# optional: connect News and Influencer means within same conflict/topic
for topic in sorted_topics:
    for conflict in ["Iran", "Afghanistan"]:
        sub = mean_df[
            (mean_df["topic_pretty"] == topic) &
            (mean_df["conflict"] == conflict)
        ].copy()

        if len(sub) == 2:
            sub["channel_type"] = pd.Categorical(sub["channel_type"], ["News", "Influencer"])
            sub = sub.sort_values("channel_type")

            ax.plot(
                sub["mean_z"],
                sub["y"],
                color=conflict_colors[conflict],
                lw=1.2,
                alpha=0.55,
                zorder=3,
            )

# ---------------------------------------------------------
# limits
# ---------------------------------------------------------
ax.set_xlim(-0.6, 0.6)
ax.set_ylim(-0.7, max(topic_to_y.values()) + 0.7)

ax.set_yticks([topic_to_y[t] for t in sorted_topics])
ax.set_yticklabels(sorted_topics, fontsize=FS_TICK, fontweight="bold")
ax.invert_yaxis()

ax.set_xlabel("Mean Topic Sentiment Z-Score", fontsize=FS_LABEL, fontweight="bold")
ax.set_ylabel("")

# ---------------------------------------------------------
# grid / styling
# ---------------------------------------------------------
ax.xaxis.set_minor_locator(AutoMinorLocator(2))
ax.grid(axis="x", which="major", linestyle="--", linewidth=0.85, alpha=0.50, color="gray")
ax.grid(axis="x", which="minor", linestyle=":", linewidth=0.55, alpha=0.30, color="gray")
ax.set_axisbelow(True)

ax.tick_params(axis="x", which="major", labelsize=FS_TICK)
ax.tick_params(axis="y", which="major", labelsize=FS_TICK, length=0)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

ax.spines["left"].set_linewidth(1.0)
ax.spines["bottom"].set_linewidth(1.0)

# ---------------------------------------------------------
# legend
# ---------------------------------------------------------
handles = [
    Line2D([0], [0], marker="o", color="none", markerfacecolor="white",
           markeredgecolor="black", markeredgewidth=1.0, markersize=8.5, label="News mean"),
    Line2D([0], [0], marker="^", color="none", markerfacecolor="white",
           markeredgecolor="black", markeredgewidth=1.0, markersize=9.0, label="Influencer mean"),
    Line2D([0], [0], marker="o", color="none", markerfacecolor=conflict_colors["Iran"],
           markeredgecolor="black", markeredgewidth=0.9, markersize=8.5, label="Iran war"),
    Line2D([0], [0], marker="o", color="none", markerfacecolor=conflict_colors["Afghanistan"],
           markeredgecolor="black", markeredgewidth=0.9, markersize=8.5, label="Afghanistan war"),
]

ax.legend(
    handles=handles,
    loc="upper right",
    frameon=True,
    fontsize=FS_LEGEND,
    ncol=2,
)

# ---------------------------------------------------------
# save figure as PDF
# ---------------------------------------------------------
output_dir = Path("images")
output_dir.mkdir(parents=True, exist_ok=True)  # cria se não existir

output_path = output_dir / "polar_words.pdf"

plt.savefig(
    output_path,
    format="pdf",
    bbox_inches="tight",
    facecolor=fig.get_facecolor(),
)

print(f"Saved figure to: {output_path.resolve()}")

plt.show()